# Did Digital Media Erode Trust in China's Courts?

### A two-wave analysis of public judicial trust · CGSS 2017 & 2023

---

**Research question.** China has rolled out one of the world's largest court
digitalisation programmes while algorithm-driven platforms have become a main
channel for judicial news. Does the spread of digital media *erode* public
trust in the judiciary?

**Data.** [China General Social Survey (CGSS)](https://cgss.ruc.edu.cn/) waves
**2017** and **2023** — nationally representative microdata, used under its
academic non-commercial licence. This notebook reads **de-identified** derived
datasets (respondent and geographic identifiers removed); the raw microdata is
not redistributed.

**What this notebook covers**

| § | Analysis |
|---|---|
| 1 | Pool the two waves, build weights and income quintiles |
| 2 | Trust levels: the headline change |
| 3 | Birth cohort — who moved the most? |
| 4 | Province-level change |
| 5 | Internet use and adjusted trust |
| 6 | Summary |

> **Measurement note.** The trust question changed between waves — an 11-point
> scale about *courts* in 2017 vs. a 4-point scale about *judges* in 2023. We
> therefore report several harmonised outcomes instead of trusting a single
> raw mean. See the project README for the full harmonisation strategy.

In [1]:
# --- 1. Setup -------------------------------------------------------------
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 130,
})

C_2017, C_2023 = "#4C72B0", "#C44E52"
FIG = Path("figures")
FIG.mkdir(exist_ok=True)

# Locate the data folder whether the notebook is run from the repo root
# or from the src/ directory.
_candidates = [Path("data"), Path("../data"), Path.cwd() / "data"]
DATA = next((p for p in _candidates if (p / "cgss_pooled_2017_2023.csv").exists()), None)
if DATA is None:
    raise FileNotFoundError("Cannot find data/cgss_pooled_2017_2023.csv — run from the repo root.")

print("pandas", pd.__version__)
print("data folder:", DATA.resolve())

pandas 3.0.6
data folder: C:\Users\26781\Desktop\CGSS\portfolio_build\data-science-project\data


## 1. Pooling the two waves

We load the two cleaned waves, append them, and derive three analysis
variables:

* **`wt_n`** — survey weights normalised *within* each year, so the two waves
  contribute equally to pooled estimates (raw weights are ~1.7× larger in 2023,
  which would otherwise let that wave dominate).
* **`incq`** — income quintile computed *within* year, because household income
  is not comparable across waves (inflation).
* **`city`** — the prefecture-city code used for clustered standard errors.

In [2]:
# --- 2. Build the pooled analysis frame -----------------------------------
d17 = pd.read_csv(DATA / "cgss2017_analysis.csv", encoding="utf-8-sig")
d23 = pd.read_csv(DATA / "cgss2023_analysis.csv", encoding="utf-8-sig")
d = pd.concat([d17, d23], ignore_index=True)

print(f"2017: {d17.shape[0]:>6} rows x {d17.shape[1]} cols")
print(f"2023: {d23.shape[0]:>6} rows x {d23.shape[1]} cols")
print(f"pooled: {d.shape[0]:>6} rows")

# weights normalised within each year
d["wt_n"] = d["wt"] * d.groupby("year")["wt"].transform("size") \
                    / d.groupby("year")["wt"].transform("sum")

# within-year income quintile; 0 = income missing
d["incq"] = 0.0
for y in d["year"].unique():
    yy = d[d["year"] == y]
    ok = yy["lninc"].notna()
    d.loc[yy.index[ok], "incq"] = (
        pd.qcut(yy.loc[ok, "lninc"], 5, labels=False, duplicates="drop") + 1)

d["y2023"] = (d["year"] == 2023).astype(float)

def wmean(x, w):
    m = pd.notna(x) & pd.notna(w)
    return float(np.average(np.asarray(x)[m], weights=np.asarray(w)[m]))

print("\nweight sums per year (normalised):")
print(d.groupby("year")["wt_n"].sum().round(1).to_string())

2017:  12582 rows x 28 cols
2023:  11326 rows x 31 cols
pooled:  23908 rows

weight sums per year (normalised):
year
2017.0    12582.0
2023.0    11326.0


## 2. Trust levels — the headline result

Trust in the judiciary rose sharply between the two waves. The binary outcome
registers the largest change because more respondents moved into the top two
categories; the standardised (POMP) score, which preserves the full spread,
records a smaller but consistent rise.

> Both outcomes are reported precisely because the scale changed between
> waves — a single number would hide that ambiguity.

In [3]:
# --- 3. Weighted trust levels by wave -------------------------------------
rows = []
m = d["trust_bin"].notna()
for y in (2017, 2023):
    s = d[m & (d.year == y)]
    rows.append({
        "year": y,
        "n": len(s),
        "trust rate (binary)": wmean(s.trust_bin, s.wt_n),
        "trust score (POMP 0-1)": wmean(s.trust_pomp, s.wt_n),
    })
levels = pd.DataFrame(rows).set_index("year")
levels["trust rate (binary)"] = (levels["trust rate (binary)"]*100).round(1)
levels["trust score (POMP 0-1)"] = levels["trust score (POMP 0-1)"].round(3)

delta_bin = levels.loc[2023, "trust rate (binary)"] - levels.loc[2017, "trust rate (binary)"]
delta_pomp = (levels.loc[2023, "trust score (POMP 0-1)"]
              - levels.loc[2017, "trust score (POMP 0-1)"])
print(levels.to_string())
print(f"\nChange 2017 -> 2023:  {delta_bin:+.1f} pp (binary)   "
      f"{delta_pomp*100:+.2f} pp (POMP)")

         n  trust rate (binary)  trust score (POMP 0-1)
year                                                   
2017  3953                 61.9                   0.646
2023  5583                 83.1                   0.694

Change 2017 -> 2023:  +21.2 pp (binary)   +4.80 pp (POMP)


In [4]:
# --- headline figure ------------------------------------------------------
fig, ax = plt.subplots(figsize=(5.4, 4.0))
vals = levels["trust rate (binary)"]
bars = ax.bar(["2017", "2023"], vals.values, width=0.55, color=[C_2017, C_2023])
for b, v in zip(bars, vals.values):
    ax.text(b.get_x()+b.get_width()/2, v+1.2, f"{v:.1f}%", ha="center",
            fontsize=11, fontweight="bold")
ax.set_ylabel("Judicial trust rate (%)")
ax.set_title("Judicial trust rose 21 points in six years")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIG / "fig_trust_levels.png", bbox_inches="tight")
plt.close(fig)
print("saved", FIG / "fig_trust_levels.png")

saved figures\fig_trust_levels.png


## 3. Birth cohort — who moved the most?

Cutting the sample by birth cohort shows the rise was **not uniform**. The
1980s cohort started as the *least* trusting group in 2017 and registered the
largest gain, so generational gaps **converged** rather than widened.

This matters for the digital-media story: if digital exposure were corroding
trust, we would expect the most-connected cohorts to fall behind — the
opposite is observed.

In [5]:
# --- 4. Trust rate by birth cohort ----------------------------------------
LAB = {1: "<=1949", 2: "1950s", 3: "1960s", 4: "1970s", 5: "1980s", 6: "1990s+"}
s = d[d["trust_bin"].notna()]
cohort = (s.groupby(["cohort", "year"])
            .apply(lambda x: wmean(x.trust_bin, x.wt_n), include_groups=False)
            .unstack("year") * 100).round(1)
cohort.index = [LAB[i] for i in cohort.index]
cohort["change (pp)"] = (cohort[2023] - cohort[2017]).round(1)
print(cohort.to_string())

x = np.arange(len(cohort)); w = 0.38
fig, ax = plt.subplots(figsize=(7.6, 4.2))
ax.bar(x-w/2, cohort[2017], w, label="2017", color=C_2017)
ax.bar(x+w/2, cohort[2023], w, label="2023", color=C_2023)
for i, (a, b) in enumerate(zip(cohort[2017], cohort[2023])):
    ax.text(i-w/2, a+0.9, f"{a:.0f}", ha="center", fontsize=8.5)
    ax.text(i+w/2, b+0.9, f"{b:.0f}", ha="center", fontsize=8.5)
ax.set_xticks(x); ax.set_xticklabels(cohort.index)
ax.set_ylim(0, 112); ax.set_yticks([0, 20, 40, 60, 80, 100])
ax.set_xlabel("Birth cohort"); ax.set_ylabel("Judicial trust rate (%)")
ax.set_title("The 1980s cohort gained the most (+30 pp)")
ax.legend(frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(); fig.savefig(FIG / "fig_cohort.png", bbox_inches="tight")
plt.close(fig); print("saved", FIG / "fig_cohort.png")

year    2017.0  2023.0  change (pp)
<=1949    67.9    81.0         13.1
1950s     61.2    78.6         17.4
1960s     61.2    82.2         21.0
1970s     62.4    85.4         23.0
1980s     55.6    85.0         29.4
1990s+    66.9    89.3         22.4


saved figures\fig_cohort.png


## 4. Province level

Coverage is lopsided — 2017 spans the whole country while the 2023 in-home
module covers fewer provinces — so we compare **only the provinces present in
both waves**. Every one of them sits **above the 45° line**, i.e. trust rose
in each. This rules out "the national average rose only because the sample
changed" and shows the gain is geographically broad.

In [6]:
# --- 5. Province-level scatter (matched provinces only) -------------------
s = d[d["trust_bin"].notna() & d["province"].notna()].copy()
prov = (s.groupby(["province", "year"])
          .apply(lambda x: wmean(x.trust_bin, x.wt_n), include_groups=False)
          .unstack("year").dropna() * 100)

fig, ax = plt.subplots(figsize=(5.6, 5.4))
ax.scatter(prov[2017], prov[2023], s=45, color=C_2023,
           edgecolor="white", linewidth=0.8, zorder=3)
lim = [40, 100]
ax.plot(lim, lim, ls="--", lw=1, color="grey", zorder=1)
ax.text(93, 88.5, "45\u00b0 line\n(no change)", fontsize=8.5, color="grey", ha="right")
ax.set_xlim(*lim); ax.set_ylim(*lim)
ax.set_xlabel("Trust rate 2017 (%)"); ax.set_ylabel("Trust rate 2023 (%)")
ax.set_title(f"Provincial trust, 2017 vs 2023 (n={len(prov)})")
ax.grid(alpha=0.25, zorder=0)
fig.tight_layout(); fig.savefig(FIG / "fig_province.png", bbox_inches="tight")
plt.close(fig)
print(f"{len(prov)} matched provinces | above the 45-degree line: "
      f"{int((prov[2023] > prov[2017]).sum())}")

16 matched provinces | above the 45-degree line: 16


## 5. Internet use and adjusted trust

Finally, the core question. We estimate a linear probability model of judicial
trust on internet-use frequency, **adjusting for** birth cohort, sex, education,
hukou, party membership, region, income quintile, social insurance and
employment status. Weights are applied and standard errors are clustered at the
prefecture-city level.

The adjusted trust probability **falls** by roughly 1 percentage point per
step up the internet-use scale. Two cautions matter for interpretation:
the association is modest, and — as the main analysis shows — it is
**significant in 2017 but not in 2023**, consistent with a narrowing rather
than widening digital divide.

In [7]:
# --- 6. Adjusted predictions by internet-use frequency --------------------
import statsmodels.api as sm

need = ["trust_bin", "internet", "cohort", "female", "edu_years", "hukou",
        "party", "region", "econ_subj", "incq", "si_any", "empstat", "s42c"]
sub = d.dropna(subset=need).copy()

terms = pd.get_dummies(
    sub[["cohort", "hukou", "region", "empstat", "incq"]],
    columns=["cohort", "hukou", "region", "empstat", "incq"],
    drop_first=True, dtype=float)
X = pd.concat([sub[["y2023", "female", "edu_years", "party", "econ_subj",
                    "si_any", "internet"]].astype(float), terms], axis=1)
X = sm.add_constant(X)

model = sm.WLS(sub["trust_bin"].astype(float), X, weights=sub["wt_n"]).fit(
    cov_type="cluster", cov_kwds={"groups": sub["s42c"]})

INTERNET = {1: "Never", 2: "Rarely", 3: "Sometimes", 4: "Often",
            5: "Very frequently"}
base = X.copy()
base["internet"] = 0
preds = []
for lvl, name in INTERNET.items():
    Xl = base.copy(); Xl["internet"] = lvl
    p = model.predict(Xl)
    preds.append({"level": lvl, "label": name, "adj. probability": p.mean()})
ip = pd.DataFrame(preds)
print(ip.round(3).to_string(index=False))

beta = model.params["internet"]
ci = model.conf_int().loc["internet"]
print(f"\nChange per one-step increase in internet use: {beta:+.2%}")
print(f"95% CI: [{ci[0]:+.2%}, {ci[1]:+.2%}]   p = {model.pvalues['internet']:.4f}")
print(f"N = {int(model.nobs)}, city clusters = {sub['s42c'].nunique()}")

fig, ax = plt.subplots(figsize=(6.6, 4.0))
ax.plot(ip["level"], ip["adj. probability"]*100, marker="o", color=C_2023, lw=2)
ax.set_xticks(list(INTERNET)); ax.set_xticklabels(list(INTERNET.values()))
ax.set_xlabel("Internet-use frequency"); ax.set_ylabel("Adjusted trust probability (%)")
ax.set_title("Adjusted judicial trust falls slightly with internet use")
ax.grid(alpha=0.25)
fig.tight_layout(); fig.savefig(FIG / "fig_internet.png", bbox_inches="tight")
plt.close(fig); print("saved", FIG / "fig_internet.png")

 level           label  adj. probability
     1           Never             0.853
     2          Rarely             0.845
     3       Sometimes             0.836
     4           Often             0.828
     5 Very frequently             0.820

Change per one-step increase in internet use: -0.81%
95% CI: [-1.80%, +0.18%]   p = 0.1078
N = 5372, city clusters = 42
saved figures\fig_internet.png


## 6. Summary

| Finding | Evidence |
|---|---|
| **Trust rose sharply** | 61.9% → 83.1% (binary, **+21 pp**); POMP +4.8 pp |
| **The rise was broadly shared** | urban–rural and education gaps statistically unchanged |
| **The 1980s cohort moved most** | +29.7 pp, the largest of any cohort; generational gaps converged |
| **Geography confirms it** | every matched province sits above the 45° line |
| **Internet use is modestly negative** | ≈ −1 pp per step, but **not significant in 2023** |

**Take-away.** Whether a court is digitalised does not determine public trust —
*how* it is digitalised does. Trust tracks subjective, procedural experience
(perceived fairness is the strongest predictor in the full model) rather than
technology exposure itself.

**Limitations.** (1) The two waves used different trust scales and referents
(courts vs. judges); the harmonisation makes the *direction* robust but not the
exact magnitude. (2) Only 16 provinces are covered in 2023, limiting
province-level precision. (3) Cross-sectional repeated data cannot identify
causal effects.

---
*Author: WANG Weicheng · MSc Data Science, University of Macau · project ongoing.*